# AcousticAcre: train the pump model on more real pumps

Today's pump model learned from **4 pumps**. This adds **6 more real pumps** from public research recordings (Hitachi MIMII, via the DCASE 2020 and 2021 challenges) and gives a fair **before/after** on the 6 new pumps: before = today's model, which never heard them; after = each new pump scored by a model trained on all the *other* pumps.

**No GPU needed.** Downloads about 1.5 GB; total time about 20–30 minutes.

1. Upload `acousticacre.zip` (the `git archive` zip) with the folder icon on the left.
2. Run the cells in order.

**Licence:** these recordings are CC BY-NC-SA 4.0, which means **non-commercial use only**. Fine for the hackathon prototype and research; a paid product needs our own recordings or permission from the dataset owners.

In [ ]:
!unzip -q -o acousticacre.zip -d acousticacre
!pip -q install onnx onnxruntime importlib_resources
%cd /content/acousticacre

## 1. Scripts

In [ ]:
%%writefile /content/acousticacre/tools/import_more_pumps.py
"""Add more real pump machines to ml/data/raw, next to the 4 DCASE 2020 development pumps.

Sources (both Hitachi MIMII recordings, CC BY-NC-SA 4.0, research / non-commercial use):
  1. DCASE 2020 Task 2 evaluation set, pump IDs 01, 03, 05 (zenodo.org/record/3841772,
     eval_data_test_pump.zip) + its ground-truth list (zenodo.org/record/3951620, eval_data_list.csv).
  2. DCASE 2021 Task 2 development set = MIMII DUE, pump sections 00, 01, 02 (zenodo.org/record/4562016,
     dev_data_pump.zip). Each section is a different pump product; labels are in the file names.

    python tools/import_more_pumps.py --eval2020 /content/eval2020 --gt eval_data_list.csv \
                                      --dev2021 /content/dev2021 --dst ml/data/raw --per-class 150

Writes ml/data/raw/{healthy,faulty}/<machine>/... with one folder per physical machine, so
leave-one-machine-out keeps holding out a whole machine at once.
"""
from __future__ import annotations

import argparse
import random
import re
import shutil
from collections import defaultdict
from pathlib import Path


def gt_pump_labels(csv_path: Path) -> dict[str, str]:
    """eval_data_list.csv: a machine-type line, then 'file,original,label' rows. Returns pump file -> label."""
    labels, section = {}, None
    for line in csv_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        if "," not in line:
            section = line
            continue
        if section == "pump":
            f, _orig, lab = line.split(",")[:3]
            labels[f] = "faulty" if lab.strip() == "1" else "healthy"
    return labels


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--eval2020", help="folder where eval_data_test_pump.zip was unzipped")
    ap.add_argument("--gt", help="eval_data_list.csv (ground truth for the 2020 evaluation set)")
    ap.add_argument("--dev2021", help="folder where dev_data_pump.zip (2021) was unzipped")
    ap.add_argument("--dst", default="ml/data/raw")
    ap.add_argument("--per-class", type=int, default=150, help="max clips per machine per label")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    groups: dict[tuple[str, str], list[Path]] = defaultdict(list)

    if a.eval2020:
        if not a.gt:
            raise SystemExit("--gt eval_data_list.csv is needed with --eval2020")
        labels = gt_pump_labels(Path(a.gt))
        pat = re.compile(r"^id_(\d+)_\d+\.wav$")
        missing = 0
        for f in Path(a.eval2020).rglob("*.wav"):
            if "pump" not in f.parts:
                continue
            m = pat.match(f.name)
            if not m:
                continue
            lab = labels.get(f.name)
            if lab is None:
                missing += 1
                continue
            groups[(lab, f"pump_id_{m.group(1)}")].append(f)
        print(f"2020 evaluation pumps: {sum(len(v) for k, v in groups.items())} labelled clips"
              + (f", {missing} without a label (skipped)" if missing else ""))

    if a.dev2021:
        pat = re.compile(r"^section_(\d+)_(source|target)_test_(normal|anomaly)_\d+\.wav$")
        n0 = sum(len(v) for v in groups.values())
        for f in Path(a.dev2021).rglob("*.wav"):
            if "pump" not in f.parts:
                continue
            m = pat.match(f.name)
            if m:
                lab = "healthy" if m.group(3) == "normal" else "faulty"
                groups[(lab, f"pump21_sec_{m.group(1)}")].append(f)
        print(f"2021 pump sections: {sum(len(v) for v in groups.values()) - n0} labelled test clips")

    if not groups:
        raise SystemExit("No pump clips found. Check the folders you passed.")

    rnd = random.Random(a.seed)
    dst = Path(a.dst)
    for (lab, machine), files in sorted(groups.items()):
        files = sorted(files)
        rnd.shuffle(files)
        out = dst / lab / machine
        out.mkdir(parents=True, exist_ok=True)
        for f in files[: a.per_class]:
            shutil.copy2(f, out / f.name)
        print(f"  {lab:8s} {machine:16s} {min(len(files), a.per_class):4d} clips")
    machines = sorted({m for _, m in groups})
    print(f"Added {len(machines)} machines: {machines}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/acousticacre/ml/eval/machine_compare.py
"""Fair before/after test for the pump model when new machines are added.

BEFORE: train on the ORIGINAL machines only (what the app ships today), score each NEW machine.
AFTER:  leave-one-machine-out over ALL machines; report the same NEW machines, each scored by a
        model trained on every other machine (never on itself).
Both numbers are on the same unseen machines, so they can be compared directly.

    python -m eval.machine_compare --data data/raw --original pump_id_00,pump_id_02,pump_id_04,pump_id_06
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from train.audio_baseline import build, logits_of, make_model, scan  # noqa: E402


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/raw")
    ap.add_argument("--original", default="pump_id_00,pump_id_02,pump_id_04,pump_id_06")
    ap.add_argument("--C", type=float, default=0.5)
    ap.add_argument("--out", default="machine_compare.json")
    a = ap.parse_args(argv)

    items = scan(Path(a.data))
    X, y, g, _rec, labels, dropped = build(items)
    orig = set(a.original.split(","))
    machines = sorted(set(g.tolist()))
    new = [m for m in machines if m not in orig]
    if not new or not (orig & set(machines)):
        sys.exit(f"Need both original and new machines. Found: {machines}")
    print(f"{len(machines)} machines ({len(orig & set(machines))} original, {len(new)} new), {len(y)} windows")

    is_orig = np.isin(g, list(orig))
    before_model = make_model(a.C).fit(X[is_orig], y[is_orig])
    pred_before = np.full_like(y, -1)
    pred_after = np.full_like(y, -1)
    for m in machines:
        te = g == m
        if m in new:
            pred_before[te] = logits_of(before_model, X[te]).argmax(1)
        tr = ~te
        pred_after[te] = logits_of(make_model(a.C).fit(X[tr], y[tr]), X[te]).argmax(1)

    on_new = np.isin(g, new)
    per = {}
    for m in new:
        k = g == m
        per[m] = {"before_accuracy": float((pred_before[k] == y[k]).mean()),
                  "after_accuracy": float((pred_after[k] == y[k]).mean()), "windows": int(k.sum())}
    res = {
        "labels": labels, "original_machines": sorted(orig & set(machines)), "new_machines": new,
        "before_macro_f1_on_new_machines": float(f1_score(y[on_new], pred_before[on_new], average="macro")),
        "after_macro_f1_on_new_machines": float(f1_score(y[on_new], pred_after[on_new], average="macro")),
        "after_macro_f1_all_machines_lomo": float(f1_score(y, pred_after, average="macro")),
        "chance_macro_f1_majority": float(f1_score(y[on_new], np.full(on_new.sum(), np.bincount(y[on_new]).argmax()), average="macro")),
        "per_new_machine": per, "dropped_by_quality": dict(dropped),
        "note": "Same unseen machines in both numbers. MIMII/DCASE data are CC BY-NC-SA 4.0 (non-commercial).",
    }
    Path(a.out).write_text(json.dumps(res, indent=2))
    print(f"\nOn the {len(new)} NEW machines (never heard in training):")
    print(f"  BEFORE (trained on original {len(orig & set(machines))}): macro-F1 {res['before_macro_f1_on_new_machines']:.2f}")
    print(f"  AFTER  (trained on all others):        macro-F1 {res['after_macro_f1_on_new_machines']:.2f}")
    print(f"  (always-guess-one-label baseline:       {res['chance_macro_f1_majority']:.2f})")
    print(f"All {len(machines)} machines, leave-one-machine-out: macro-F1 {res['after_macro_f1_all_machines_lomo']:.2f}")
    for m, v in per.items():
        print(f"    {m:16s} before {v['before_accuracy']:.0%}  after {v['after_accuracy']:.0%}  ({v['windows']} windows)")
    print(f"saved {a.out}")


if __name__ == "__main__":
    main()


## 2. The original 4 pumps (same as the first training)
If `ls` shows a tiny file, the download failed: open the Zenodo link in a browser, copy the file's download link and paste it in.

In [ ]:
!wget -q -O /content/pump2020dev.zip "https://zenodo.org/records/3678171/files/dev_data_pump.zip?download=1"
!ls -lh /content/pump2020dev.zip
!unzip -q -o /content/pump2020dev.zip -d /content/dev2020
!python tools/import_dcase.py /content/dev2020 ml/data/raw --per-class 150 --copy

## 3. The 6 new pumps

In [ ]:
!wget -q -O /content/pump2020eval.zip "https://zenodo.org/records/3841772/files/eval_data_test_pump.zip?download=1"
!wget -q -O /content/eval_data_list.csv "https://zenodo.org/records/3951620/files/eval_data_list.csv?download=1"
!wget -q -O /content/pump2021dev.zip "https://zenodo.org/records/4562016/files/dev_data_pump.zip?download=1"
!ls -lh /content/pump2020eval.zip /content/eval_data_list.csv /content/pump2021dev.zip

In [ ]:
!unzip -q -o /content/pump2020eval.zip -d /content/eval2020
!unzip -q -o /content/pump2021dev.zip -d /content/dev2021
!python tools/import_more_pumps.py --eval2020 /content/eval2020 --gt /content/eval_data_list.csv --dev2021 /content/dev2021 --dst ml/data/raw --per-class 150

## 4. BEFORE vs AFTER on the 6 new pumps
Both numbers are on the same 6 pumps, which neither model trained on.

In [ ]:
%cd /content/acousticacre/ml
!python -m eval.machine_compare --data data/raw --out machine_compare.json

## 5. Train the new pump model on all 10 pumps and build the pack
This also prints the leave-one-machine-out result over all 10 pumps (each pump scored by a model that never heard it).

In [ ]:
%cd /content/acousticacre/ml
!python -m train.audio_baseline --data data/raw --out runs/baseline10
!python -m export.baseline_to_onnx --run runs/baseline10
!python -c "from dsp.logmel import export_constants; export_constants('runs/baseline10/dsp_constants.json')"
%cd /content/acousticacre
!python tools/pack_builder.py machine --run ml/runs/baseline10 --out packs --version 0.2.0

## 6. Download
Send `machine_compare.json` and `machine_report_10pumps.json` back to Claude. **Only put the new pack in the app if AFTER beats BEFORE.**

In [ ]:
import shutil, glob
from google.colab import files
shutil.copy('/content/acousticacre/ml/runs/baseline10/report.json', '/content/machine_report_10pumps.json')
files.download('/content/acousticacre/ml/machine_compare.json')
files.download('/content/machine_report_10pumps.json')
for f in glob.glob('/content/acousticacre/packs/machine-*-0.2.0.aapack'):
    files.download(f)